<a href="https://www.kaggle.com/code/ilginkarakas/rogii-wellbore-geology-prediction?scriptVersionId=335070990" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:

import gc
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

warnings.filterwarnings("ignore")
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/AI_wellbore_geology_prediction_task_en.pptx
/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/000d7d20__typewell.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/00e12e8b__typewell.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/00bbac68__typewell.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/000d7d20__horizontal_well.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/00bbac68__horizontal_well.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/00e12e8b__horizontal_well.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/train/75d20a82__typewell.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/train/b0f53bf4__horizontal_well.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/train/e47

In [2]:
ROOT      = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
TRAIN_DIR = f"{ROOT}/train"
TEST_DIR  = f"{ROOT}/test"
 
FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
 
sample_sub = pd.read_csv(f"{ROOT}/sample_submission.csv")
print(f"Sample submission shape: {sample_sub.shape}")
print(sample_sub.head())

Sample submission shape: (14151, 2)
              id  tvt
0  000d7d20_1442  0.0
1  000d7d20_1443  0.0
2  000d7d20_1444  0.0
3  000d7d20_1445  0.0
4  000d7d20_1446  0.0


In [3]:
train_wells = sorted({
    f.split("__")[0] for f in os.listdir(TRAIN_DIR)
    if "__horizontal_well.csv" in f
})
test_wells = sorted({
    f.split("__")[0] for f in os.listdir(TEST_DIR)
    if "__horizontal_well.csv" in f
})
print(f"\nTrain wells: {len(train_wells)}, Test wells: {len(test_wells)}")


Train wells: 773, Test wells: 3


In [4]:
def load_typewell(path):
    """Load typewell, return (clean_df, stats_dict). Handles multiple formats."""
    if not os.path.exists(path):
        return pd.DataFrame(), {}
    tw = pd.read_csv(path)
    
    # Normalize column names — some typewells use TVT_input, some TVT
    if "TVT" not in tw.columns and "TVT_input" in tw.columns:
        tw["TVT"] = tw["TVT_input"]
    
    stats = {}
    if "GR" in tw.columns:
        gr = tw["GR"].dropna()
        if len(gr):
            stats["tw_GR_mean"] = gr.mean()
            stats["tw_GR_std"]  = gr.std()
            stats["tw_GR_p10"]  = gr.quantile(0.10)
            stats["tw_GR_p50"]  = gr.quantile(0.50)
            stats["tw_GR_p90"]  = gr.quantile(0.90)
    if "TVT" in tw.columns:
        tvt = tw["TVT"].dropna()
        if len(tvt):
            stats["tw_TVT_min"]   = tvt.min()
            stats["tw_TVT_max"]   = tvt.max()
            stats["tw_TVT_range"] = tvt.max() - tvt.min()
    
    if "GR" in tw.columns and "TVT" in tw.columns:
        return tw.dropna(subset=["GR", "TVT"]).copy(), stats
    return tw, stats

In [5]:
def add_typewell_lookup(hw_df, tw_clean):
    if len(tw_clean) == 0 or "GR" not in hw_df.columns:
        return hw_df
    tw_sorted = tw_clean.sort_values("GR").reset_index(drop=True)
    tw_gr  = tw_sorted["GR"].to_numpy()
    tw_tvt = tw_sorted["TVT"].to_numpy()
    hw_gr = hw_df["GR"].fillna(tw_gr.mean()).to_numpy()
    K = 10
    idx = np.searchsorted(tw_gr, hw_gr)
    means, stds = [], []
    for i in range(len(hw_gr)):
        lo = max(0, idx[i] - K)
        hi = min(len(tw_tvt), idx[i] + K + 1)
        neigh = tw_tvt[lo:hi]
        if len(neigh):
            means.append(neigh.mean())
            stds.append(neigh.std() if len(neigh) > 1 else 0)
        else:
            means.append(np.nan)
            stds.append(np.nan)
    hw_df["tw_lookup_mean"] = means
    hw_df["tw_lookup_std"]  = stds
    return hw_df
 

In [6]:
def add_baseline(df):
    df = df.copy()
    # Forward-fill TVT_input → last known TVT value at every row
    df["tvt_baseline"] = df["TVT_input"].ffill()
    
    # If very first rows have no known TVT_input yet, use median
    if df["tvt_baseline"].isna().any():
        # Use the well's overall median TVT as fallback (train has TVT known)
        # For test, use nearest known ffill/bfill
        df["tvt_baseline"] = df["tvt_baseline"].bfill()
    
    # Also track Z at the last known point (helps understand elevation change)
    df["Z_at_last_known"] = df["Z"].where(df["TVT_input"].notna()).ffill().bfill()
    df["dZ_from_last_known"] = df["Z"] - df["Z_at_last_known"]
    
    # Distance (rows) from last known point
    is_known = df["TVT_input"].notna().to_numpy()
    dist = np.zeros(len(df), dtype=np.int32)
    counter = 0
    for i in range(len(df)):
        if is_known[i]:
            counter = 0
        else:
            counter += 1
        dist[i] = counter
    df["rows_since_known"] = dist
    
    # MD gap from last known
    df["MD_at_last_known"] = df["MD"].where(df["TVT_input"].notna()).ffill().bfill()
    df["MD_from_last_known"] = df["MD"] - df["MD_at_last_known"]
    
    return df

In [7]:
def create_features(df, tw_stats=None, tw_clean=None):
    df = df.copy().reset_index(drop=True)
    df = add_baseline(df)
    for col in ["X", "Y", "Z", "MD"]:
        if col in df.columns:
            df[f"{col}_diff1"]  = df[col].diff().fillna(0)
            df[f"{col}_diff5"]  = df[col].diff(5).fillna(0)
            df[f"{col}_diff20"] = df[col].diff(20).fillna(0)
    
    if all(c in df.columns for c in ["X", "Y", "Z"]):
        df["step_3d"] = np.sqrt(
            df["X_diff1"]**2 + df["Y_diff1"]**2 + df["Z_diff1"]**2
        )
        df["step_2d"] = np.sqrt(df["X_diff1"]**2 + df["Y_diff1"]**2)
        df["inclination"] = np.arctan2(
            df["step_2d"], np.abs(df["Z_diff1"]) + 1e-6
        )
        df["azimuth"] = np.arctan2(df["Y_diff1"], df["X_diff1"] + 1e-6)
        df["dogleg"] = (
            df["inclination"].diff().abs().fillna(0) +
            df["azimuth"].diff().abs().fillna(0)
        )
        df["cum_step3d"] = df["step_3d"].cumsum()
    
    # ---------- 6.3 FORMATION DISTANCES (CRITICAL) ----------
    if "Z" in df.columns:
        for f in FORMATIONS:
            if f in df.columns:
                df[f"Z_minus_{f}"] = df["Z"] - df[f]
        
        # Formation membership: which formation are we in?
        # Sort formations by depth at each row and find where Z falls
        if all(f in df.columns for f in FORMATIONS):
            # ANCC is shallowest, BUDA is deepest
            # Formation order: ANCC, ASTNU, ASTNL, EGFDU, EGFDL, BUDA
            # (going deeper each step)
            f_matrix = df[FORMATIONS].to_numpy()
            z_vals = df["Z"].to_numpy().reshape(-1, 1)
            # Which formation top is closest?
            dists = np.abs(f_matrix - z_vals)
            df["closest_formation"]      = np.argmin(dists, axis=1)
            df["closest_formation_dist"] = np.min(dists, axis=1)
            df["min_signed_dist"] = np.min(z_vals - f_matrix, axis=1)
            df["max_signed_dist"] = np.max(z_vals - f_matrix, axis=1)
        
        # Formation thickness (top pairs)
        pairs = [("ANCC","ASTNU"), ("ASTNU","ASTNL"), ("ASTNL","EGFDU"),
                 ("EGFDU","EGFDL"), ("EGFDL","BUDA")]
        for top, bot in pairs:
            if top in df.columns and bot in df.columns:
                df[f"thickness_{top}_{bot}"] = df[top] - df[bot]
    
    # ---------- 6.4 GR FEATURES ----------
    if "GR" in df.columns:
        gr_filled = df["GR"].ffill().bfill()
        df["GR_was_nan"] = df["GR"].isna().astype(int)
        
        # Lags & leads
        for lag in [1, 2, 3, 5, 10, 20, 50, 100]:
            df[f"GR_lag_{lag}"]  = gr_filled.shift(lag)
            df[f"GR_lead_{lag}"] = gr_filled.shift(-lag)
        
        # Rolling stats
        for w in [3, 5, 10, 20, 50, 100, 200]:
            roll = gr_filled.rolling(w, min_periods=1)
            df[f"GR_mean_{w}"]   = roll.mean()
            df[f"GR_std_{w}"]    = roll.std()
            df[f"GR_min_{w}"]    = roll.min()
            df[f"GR_max_{w}"]    = roll.max()
            df[f"GR_median_{w}"] = roll.median()
            df[f"GR_range_{w}"]  = df[f"GR_max_{w}"] - df[f"GR_min_{w}"]
        
        # Centered rolling
        for w in [10, 20, 50]:
            roll_c = gr_filled.rolling(w, min_periods=1, center=True)
            df[f"GR_cmean_{w}"] = roll_c.mean()
            df[f"GR_cstd_{w}"]  = roll_c.std()
        
        # EMA
        for span in [5, 20, 50, 100]:
            df[f"GR_ema_{span}"] = gr_filled.ewm(span=span, adjust=False).mean()
        
        # Differences
        for d in [1, 3, 5, 10, 20, 50]:
            df[f"GR_diff_{d}"] = gr_filled.diff(d)
        
        # Deviation & zscore
        df["GR_dev_20"]    = gr_filled - df["GR_mean_20"]
        df["GR_dev_50"]    = gr_filled - df["GR_mean_50"]
        df["GR_zscore_20"] = np.where(
            df["GR_std_20"] > 1e-6,
            (gr_filled - df["GR_mean_20"]) / df["GR_std_20"], 0
        )
        df["GR_zscore_50"] = np.where(
            df["GR_std_50"] > 1e-6,
            (gr_filled - df["GR_mean_50"]) / df["GR_std_50"], 0
        )
        
        df["GR"] = gr_filled
    
    # ---------- 6.5 TVT_INPUT FEATURES ----------
    if "TVT_input" in df.columns:
        df["TVT_input_known"] = df["TVT_input"].notna().astype(int)
        # Rolling of TVT_input for smoothing
        for w in [5, 10, 20, 50]:
            df[f"TVT_input_mean_{w}"] = df["TVT_input"].rolling(
                w, min_periods=1
            ).mean()
        # Rate of change of TVT_input in the known portion (extrapolated)
        df["TVT_input_diff1"]  = df["tvt_baseline"].diff().fillna(0)
        df["TVT_input_diff20"] = df["tvt_baseline"].diff(20).fillna(0)
    
    # ---------- 6.6 WELL-LEVEL STATIC ----------
    if "GR" in df.columns:
        df["well_GR_mean"] = df["GR"].mean()
        df["well_GR_std"]  = df["GR"].std()
    if "MD" in df.columns:
        df["well_MD_length"] = df["MD"].iloc[-1] - df["MD"].iloc[0]
    df["well_n_rows"] = len(df)
    df["row_pct"] = np.arange(len(df)) / max(len(df) - 1, 1)
    
    # ---------- 6.7 TYPEWELL STATIC + LOOKUP ----------
    if tw_stats:
        for k, v in tw_stats.items():
            df[k] = v
    if tw_clean is not None and len(tw_clean) > 0:
        df = add_typewell_lookup(df, tw_clean)
        # Feature: difference between typewell lookup TVT and baseline
        df["tw_lookup_minus_baseline"] = df["tw_lookup_mean"] - df["tvt_baseline"]
    
    return df

In [8]:
print("\n=== LOADING TRAIN ===")
all_train = []
for well in tqdm(train_wells, desc="Train"):
    tw_clean, tw_stats = load_typewell(f"{TRAIN_DIR}/{well}__typewell.csv")
    try:
        df = pd.read_csv(f"{TRAIN_DIR}/{well}__horizontal_well.csv")
        df["well"] = well
        df = create_features(df, tw_stats, tw_clean)
        all_train.append(df)
    except Exception as e:
        print(f"  ERROR {well}: {e}")
train = pd.concat(all_train, ignore_index=True)
del all_train; gc.collect()
print(f"Train shape: {train.shape}")
 
print("\n=== LOADING TEST ===")
all_test = []
for well in tqdm(test_wells, desc="Test"):
    tw_clean, tw_stats = load_typewell(f"{TEST_DIR}/{well}__typewell.csv")
    df = pd.read_csv(f"{TEST_DIR}/{well}__horizontal_well.csv")
    df["well"] = well
    df["row_id"] = np.arange(len(df))
    df = create_features(df, tw_stats, tw_clean)
    all_test.append(df)
test = pd.concat(all_test, ignore_index=True)
del all_test; gc.collect()
print(f"Test shape: {test.shape}")


=== LOADING TRAIN ===


Train: 100%|██████████| 773/773 [02:57<00:00,  4.36it/s]


Train shape: (5092255, 155)

=== LOADING TEST ===


Test: 100%|██████████| 3/3 [00:00<00:00,  4.94it/s]


Test shape: (19221, 134)


In [9]:
# Instead of predicting TVT (range ~11000), predict TVT - baseline (range ~10-100)
train["residual"] = train["TVT"] - train["tvt_baseline"]
print(f"\nResidual stats (training target):")
print(train["residual"].describe())
print(f"\nOriginal TVT range: {train['TVT'].min():.1f} to {train['TVT'].max():.1f}")
print(f"Residual range:     {train['residual'].min():.3f} to {train['residual'].max():.3f}")
 
# Drop rows with NaN residual (shouldn't happen if baseline is always defined)
n_before = len(train)
train = train.dropna(subset=["residual"]).reset_index(drop=True)
print(f"Dropped {n_before - len(train)} rows with NaN residual")


Residual stats (training target):
count    5.092255e+06
mean     1.185957e+00
std      1.366332e+01
min     -1.037800e+02
25%     -3.290000e+00
50%      0.000000e+00
75%      5.920000e+00
max      9.892000e+01
Name: residual, dtype: float64

Original TVT range: 9245.2 to 12893.9
Residual range:     -103.780 to 98.920
Dropped 0 rows with NaN residual


In [10]:
EXCLUDE = {"TVT", "residual", "well", "row_id", "id", "TVT_input"}
features = [
    c for c in train.columns
    if c in test.columns
    and c not in EXCLUDE
    and train[c].dtype in [np.float64, np.float32, np.int64, np.int32]
]
# tvt_baseline is a FEATURE we want (not excluded) — it's the anchor point
print(f"\nNumber of features: {len(features)}")
print(f"Includes tvt_baseline: {'tvt_baseline' in features}")


Number of features: 131
Includes tvt_baseline: True


In [11]:
medians = train[features].median()
train[features] = train[features].fillna(medians)
test[features]  = test[features].fillna(medians)
for col in features:
    train.loc[~np.isfinite(train[col]), col] = medians[col]
    test.loc[~np.isfinite(test[col]),  col] = medians[col]
train[features] = train[features].astype(np.float32)
test[features]  = test[features].astype(np.float32)
gc.collect()

0

In [12]:
print("\n=== CROSS-VALIDATION (GroupKFold, 5 folds) ===")
N_FOLDS = 5
gkf = GroupKFold(n_splits=N_FOLDS)
 
X = train[features].to_numpy()
y_residual = train["residual"].to_numpy()
y_baseline = train["tvt_baseline"].to_numpy()
y_tvt      = train["TVT"].to_numpy()
grps       = train["well"].to_numpy()
 
oof_residual = np.zeros(len(train), dtype=np.float32)
 
lgb_params = dict(
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    objective="regression",
    metric="rmse",
)
 
fold_rmse_tvt = []
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y_residual, groups=grps)):
    print(f"\n--- Fold {fold + 1}/{N_FOLDS} ---")
    m = LGBMRegressor(**lgb_params)
    m.fit(
        X[tr_idx], y_residual[tr_idx],
        eval_set=[(X[va_idx], y_residual[va_idx])],
        callbacks=[
            early_stopping(stopping_rounds=150, verbose=False),
            log_evaluation(period=0),
        ],
    )
    pred_res = m.predict(X[va_idx])
    oof_residual[va_idx] = pred_res
    
    # Recover TVT and evaluate in TVT space
    pred_tvt = y_baseline[va_idx] + pred_res
    rmse_tvt = np.sqrt(mean_squared_error(y_tvt[va_idx], pred_tvt))
    rmse_res = np.sqrt(mean_squared_error(y_residual[va_idx], pred_res))
    print(f"Fold {fold + 1}: residual RMSE={rmse_res:.4f}, TVT RMSE={rmse_tvt:.4f}")
    fold_rmse_tvt.append(rmse_tvt)
 
oof_tvt = y_baseline + oof_residual
cv_rmse_tvt = np.sqrt(mean_squared_error(y_tvt, oof_tvt))
print(f"\n*** CV RMSE in TVT space: {cv_rmse_tvt:.4f} ***")
print(f"Fold mean: {np.mean(fold_rmse_tvt):.4f} ± {np.std(fold_rmse_tvt):.4f}")
 
# Compare to naive baseline
naive_rmse = np.sqrt(mean_squared_error(y_tvt, y_baseline))
print(f"\nNaive baseline (just use last known TVT): {naive_rmse:.4f}")
print(f"Model improvement: {naive_rmse - cv_rmse_tvt:.4f}")


=== CROSS-VALIDATION (GroupKFold, 5 folds) ===

--- Fold 1/5 ---
Fold 1: residual RMSE=11.4472, TVT RMSE=11.4472

--- Fold 2/5 ---
Fold 2: residual RMSE=13.2995, TVT RMSE=13.2995

--- Fold 3/5 ---
Fold 3: residual RMSE=13.1844, TVT RMSE=13.1844

--- Fold 4/5 ---
Fold 4: residual RMSE=11.3452, TVT RMSE=11.3452

--- Fold 5/5 ---
Fold 5: residual RMSE=13.8132, TVT RMSE=13.8132

*** CV RMSE in TVT space: 12.6593 ***
Fold mean: 12.6179 ± 1.0202

Naive baseline (just use last known TVT): 13.7147
Model improvement: 1.0554


In [13]:
importance_df = pd.DataFrame({
    "feature": features,
    "importance": m.feature_importances_
}).sort_values("importance", ascending=False)
print("\n=== TOP 25 FEATURES ===")
print(importance_df.head(25).to_string(index=False))


=== TOP 25 FEATURES ===
           feature  importance
dZ_from_last_known       28962
           row_pct       26462
  rows_since_known       24543
                MD       22297
                 Y       19569
                 Z       19481
        cum_step3d       19188
                 X       17049
       well_GR_std       13468
        GR_max_200       12962
    well_MD_length       12634
        GR_min_200       12379
      well_GR_mean       12244
          Z_diff20       10867
  MD_at_last_known       10619
      tw_TVT_range       10609
         tw_GR_std        9603
      tvt_baseline        9340
         tw_GR_p10        8867
      GR_range_200        8541
           Z_diff5        8479
          Y_diff20        8114
          X_diff20        8067
        tw_GR_mean        8003
     GR_median_200        7399


In [14]:
print("\n=== FINAL LIGHTGBM (multi-seed) ===")
X_train = train[features].to_numpy()
y_train_res = train["residual"].to_numpy()
X_test = test[features].to_numpy()
 
test_pred_res_lgb = np.zeros(len(test), dtype=np.float32)
SEEDS = [42, 123, 2024, 7, 11]
 
for seed in SEEDS:
    print(f"Seed {seed}...")
    p = lgb_params.copy()
    p["random_state"] = seed
    m = LGBMRegressor(**p)
    m.fit(X_train, y_train_res)
    test_pred_res_lgb += m.predict(X_test) / len(SEEDS)


=== FINAL LIGHTGBM (multi-seed) ===
Seed 42...
Seed 123...
Seed 2024...
Seed 7...
Seed 11...


In [15]:
try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except ImportError:
    HAS_CAT = False
    print("CatBoost unavailable — using LGB only")
 
if HAS_CAT:
    print("\n=== CATBOOST ===")
    cat_params = dict(
        iterations=3000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=42,
        verbose=0,
        early_stopping_rounds=150,
    )
    
    # Quick single-fold check
    tr_q, va_q = next(gkf.split(X, y_residual, groups=grps))
    cat_q = CatBoostRegressor(**cat_params)
    cat_q.fit(
        X[tr_q], y_residual[tr_q],
        eval_set=(X[va_q], y_residual[va_q])
    )
    cat_pred_res = cat_q.predict(X[va_q])
    cat_pred_tvt = y_baseline[va_q] + cat_pred_res
    cat_rmse_tvt = np.sqrt(mean_squared_error(y_tvt[va_q], cat_pred_tvt))
    print(f"CatBoost single-fold TVT RMSE: {cat_rmse_tvt:.4f}")
    
    # Final training
    cat_final = CatBoostRegressor(**cat_params)
    cat_final.fit(X_train, y_train_res)
    test_pred_res_cat = cat_final.predict(X_test)
    
    # Ensemble: weighted average
    test_pred_res = 0.6 * test_pred_res_lgb + 0.4 * test_pred_res_cat
else:
    test_pred_res = test_pred_res_lgb


=== CATBOOST ===
CatBoost single-fold TVT RMSE: 11.2303


In [16]:
test_baseline = test["tvt_baseline"].to_numpy()
test_pred_tvt = test_baseline + test_pred_res
 
# SAFETY: if predicted residual is unreasonable, fallback to baseline
# (This prevents catastrophic errors on out-of-distribution rows)
residual_std_train = train["residual"].std()
outlier_mask = np.abs(test_pred_res) > 10 * residual_std_train
if outlier_mask.any():
    print(f"WARNING: {outlier_mask.sum()} extreme predictions clamped to baseline")
    test_pred_tvt[outlier_mask] = test_baseline[outlier_mask]
 
test["pred"] = test_pred_tvt
 
print(f"\nFinal predictions:")
print(f"  Range: [{test_pred_tvt.min():.2f}, {test_pred_tvt.max():.2f}]")
print(f"  Mean:  {test_pred_tvt.mean():.2f}")


Final predictions:
  Range: [10606.43, 12238.34]
  Mean:  11830.63


In [17]:
test["id"] = test["well"] + "_" + test["row_id"].astype(str)
submission = sample_sub[["id"]].merge(
    test[["id", "pred"]], on="id", how="left"
)
submission = submission.rename(columns={"pred": "tvt"})
 
missing = submission["tvt"].isna().sum()
print(f"\nUnmatched: {missing}")
if missing > 0:
    print("Filling with median baseline")
    submission["tvt"] = submission["tvt"].fillna(train["TVT"].median())
 
# CRITICAL sanity check — most predictions should be similar in magnitude to TVT_input
print(f"\nSubmission stats:")
print(submission["tvt"].describe())
print(f"Unique values: {submission['tvt'].nunique()}")
 
submission.to_csv("submission.csv", index=False)
print(f"\n✓ Saved: submission.csv ({len(submission)} rows)")
print(submission.head(10))
 


Unmatched: 0

Submission stats:
count    14151.000000
mean     11903.611596
std        278.024436
min      11588.952585
25%      11612.790453
50%      11745.072787
75%      12226.459846
max      12238.335840
Name: tvt, dtype: float64
Unique values: 14151

✓ Saved: submission.csv (14151 rows)
              id           tvt
0  000d7d20_1442  11746.921349
1  000d7d20_1443  11746.952773
2  000d7d20_1444  11747.152152
3  000d7d20_1445  11747.108661
4  000d7d20_1446  11747.172941
5  000d7d20_1447  11747.225880
6  000d7d20_1448  11747.301783
7  000d7d20_1449  11747.347811
8  000d7d20_1450  11747.359506
9  000d7d20_1451  11747.304215
